# XGBoost·CatBoost 평가 지표 및 비교 결론 — demo2


## 먼저 알아야 할 데이터

- 한 행: 학생 1명 × 과목 1개 × 운영회차 1개 × 예측 주차 1개
- 사용 주차: **4·19·25주차**
- 전체 행: **75,125행**
- 다음 7일 안 이탈: **588건 (0.7827%)**
- 모델 입력: **98개 Feature**, 범주형 8개
- 검증: 같은 학생이 학습과 평가에 섞이지 않는 `id_student` 기준 3-Fold GroupKFold

> 주의: 여기서 예측하는 것은 '언젠가 최종 이탈하는가'가 아니라 **바로 다음 7일 안에 이탈하는가**입니다.


## 평가 지표를 쉬운 말로 이해하기

| 지표 | 한 문장 설명 | 좋은 방향 |
|---|---|---|
| `Recall@Top-20%` | 위험도가 높은 상위 20% 안에 실제 이탈자를 얼마나 담았는가 | 높을수록 좋음 |
| `PR-AUC` | 매우 적은 이탈자를 비이탈자와 얼마나 잘 구분하는가 | 높을수록 좋음 |
| `Brier Score` | 예측확률과 실제 0·1 결과의 평균 제곱 오차 | 낮을수록 좋음 |
| `ECE` | '예측 10%' 집단이 실제로도 약 10% 이탈하는지 보는 오차 | 낮을수록 좋음 |

정확도(Accuracy)는 대부분이 비이탈자인 데이터에서 높게 나오기 쉬우므로 핵심 판단 지표로 사용하지 않습니다.


## 실제 숫자로 이해하기

위험 상위 20%는 약 **15,025행**입니다.

- XGBoost는 전체 이탈 588건 중 약 **411건**을 상위 20% 안에서 찾았습니다.
- CatBoost는 전체 이탈 588건 중 약 **402건**을 상위 20% 안에서 찾았습니다.
- XGBoost가 약 **9건** 더 찾았지만, 이것만으로 최종 모델을 고르면 안 됩니다.

확률 평균도 실제 이탈률과 비교해야 합니다.

- 실제 이탈률: **0.783%**
- XGBoost 평균 예측확률: **27.12%**
- CatBoost 평균 예측확률: **0.770%**

XGBoost는 `scale_pos_weight`로 이탈자를 강하게 학습했기 때문에 원시 확률이 실제 이탈률보다 크게 부풀 수 있습니다.


## 모델별 전체 평가 결과

| 모델 | Recall@Top-20% ↑ | PR-AUC ↑ | Brier ↓ | ECE ↓ |
|---|---:|---:|---:|---:|
| XGBoost | **0.6990** | 0.0396 | 0.113504 | 0.263375 |
| CatBoost | 0.6837 | **0.0438** | **0.007615** | **0.000150** |

### 해석

- XGBoost는 위험 상위 20%에서 이탈자를 조금 더 많이 찾았습니다.
- CatBoost는 PR-AUC가 더 높아 전체 순위 구분력이 더 좋습니다.
- CatBoost의 Brier와 ECE가 압도적으로 낮아 원시 확률이 실제 이탈률에 훨씬 가깝습니다.
- 따라서 위험 순위와 확률 표시를 함께 고려하면 CatBoost가 더 균형 잡힌 결과입니다.


## Fold별 성능 안정성

| 모델 | Fold 1 PR-AUC | Fold 2 PR-AUC | Fold 3 PR-AUC |
|---|---:|---:|---:|
| XGBoost | 0.0466 | 0.0460 | 0.0513 |
| CatBoost | 0.0433 | 0.0508 | 0.0449 |

Fold별로는 승자가 섞여 있습니다. XGBoost가 Fold 1·3, CatBoost가 Fold 2에서 높습니다. 따라서 한 Fold만 보고 결론 내리지 않고 전체 OOF 결과와 확률 품질을 함께 봅니다.


## 결론 — 무엇을 선택해야 하나요?

**현재 demo2 비교 결과의 1순위는 CatBoost입니다.**

이유는 다음과 같습니다.

1. XGBoost의 Recall 이점은 실제 이탈자 약 9건 차이로 제한적입니다.
2. CatBoost의 PR-AUC가 더 높습니다.
3. CatBoost의 Brier와 ECE가 훨씬 낮아 확률을 해석하기 쉽습니다.
4. 범주형 Feature를 One-Hot으로 펼치지 않아 코드와 배포 구조도 단순합니다.

XGBoost는 보조 비교 모델과 위험 순위 모델로 유지할 가치가 있습니다. 하지만 현재 원시 확률은 사용자 화면에 바로 표시하지 말고, 별도 검증 데이터에서 Platt 보정 등을 적용한 뒤 다시 평가해야 합니다.

> 가장 중요한 제한: 이 데이터는 4·19·25주차만 포함합니다. '매주 갱신 서비스'를 완성하려면 모든 운영 주차 데이터로 다시 학습해야 합니다.
